## UrbanRide_08 — Gold Trip Features
**Method:** Batch enrichment — Silver → Gold  
**Inputs:**
- `urbanride.silver.trips` — base trip data (21M+ rows)
- `urbanride.silver.drivers` — driver attributes (20K rows — broadcast)
- `urbanride.silver.customers` — customer attributes (89K rows — broadcast)

**Output:** `urbanride.gold.trip_features`  
**Schedule:** Daily · runs after Silver layer is complete  
**Partition:** `trip_status`  
**ZORDER:** `trip_date`, `city`

**Grain:** one row per `trip_id`

**Feature engineering:**
- Direct trip attributes from silver.trips
- Derived: `trip_date`, `fare_per_km`, `is_long_trip`, `is_peak_surge`
- Driver enrichment via broadcast join: rating, status, is_high_value_driver
- Customer enrichment via broadcast join: city, device_type, is_churned

**Feeds:** Fraud Detection Model (12) · Demand Forecasting Model (13)

In [0]:
from pyspark.sql.functions import (
    col, when, lit, current_timestamp,
    round, broadcast
)

CATALOG = 'urbanride'
SILVER  = f'{CATALOG}.silver'
GOLD    = f'{CATALOG}.gold'

spark.sql(f'USE CATALOG {CATALOG}')

print(f'Catalog : {CATALOG}')
print(f'Sources : {SILVER}.trips, {SILVER}.drivers, {SILVER}.customers')
print(f'Target  : {GOLD}.trip_features')
print('Config ready.')


Catalog : urbanride
Sources : urbanride.silver.trips, urbanride.silver.drivers, urbanride.silver.customers
Target  : urbanride.gold.trip_features
Config ready.


In [0]:
# Gold trip_features uses incremental append — unlike customer_features
# Reason: trip rows are immutable once written to Silver
# A trip that happened on Day 1 never changes — no date-based recalculations needed
# Strategy: append only new trips ingested since last Gold run
# Watermark: silver.trips._processed_at vs gold.trip_features._processed_at

print('Checking load type...')

table_exists = spark.catalog.tableExists(f'{GOLD}.trip_features')

if not table_exists:
    LOAD_TYPE = 'full'
    last_run  = None
    print('  Gold table does not exist — FULL LOAD')
else:
    LOAD_TYPE = 'incremental'
    last_run  = spark.sql(
        f'SELECT MAX(_processed_at) FROM {GOLD}.trip_features'
    ).collect()[0][0]
    print(f'  Gold table exists — INCREMENTAL LOAD')
    print(f'  Last processed at : {last_run}')

print(f'  Load type : {LOAD_TYPE}')


Checking load type...
  Gold table does not exist — FULL LOAD
  Load type : full


In [0]:
print('Reading Silver tables...')

# Trips — filter by watermark for incremental
df_trips_full = spark.table(f'{SILVER}.trips')

if LOAD_TYPE == 'full':
    df_trips = df_trips_full
    print('  Full load — reading all trip rows')
else:
    df_trips = df_trips_full.filter(col('_processed_at') > last_run)
    print(f'  Incremental — reading trips processed after {last_run}')

# Drivers and customers — always read full for enrichment
# These are small lookup tables — always need latest attributes
df_drivers   = spark.table(f'{SILVER}.drivers')
df_customers = spark.table(f'{SILVER}.customers')

trip_count = df_trips.count()
print(f'  Trips to process : {trip_count:,}')

if trip_count == 0:
    print('  No new trips to process — Gold already up to date.')
    dbutils.notebook.exit('No new rows — skipping')

print(f'  Drivers          : {df_drivers.count():,}')
print(f'  Customers        : {df_customers.count():,}')


Reading Silver tables...
  Full load — reading all trip rows
  Trips to process : 21,359,745
  Drivers          : 20,000
  Customers        : 89,041


In [0]:
print('Building trip base features...')

df_trip_base = df_trips.select(
    col('trip_id'),
    col('customer_id'),
    col('driver_id'),
    col('city'),
    col('trip_status'),
    col('pickup_time'),
    col('drop_time'),
    col('pickup_time').cast('date').alias('trip_date'),
    col('distance_km'),
    col('fare_amount'),
    col('surge_multiplier'),
    col('trip_duration_minutes'),
    col('vehicle_type'),
    col('weather'),
    col('day_type'),
    col('is_ghost_trip'),
    col('is_fare_inflated'),
    col('is_distance_invalid'),
    col('is_duration_invalid'),
    col('is_status_invalid'),

    # Derived: fare_per_km — detect fare inflation relative to distance
    # Null-safe: avoid division by zero on ghost trips (distance = 0)
    when(
        col('distance_km') > 0,
        round(col('fare_amount') / col('distance_km'), 2)
    ).otherwise(None).alias('fare_per_km'),

    # Derived: is_long_trip — distance > 20km
    # Useful for demand forecasting — long trips behave differently
    when(col('distance_km') > 15, True).otherwise(False).alias('is_long_trip'),

    # Derived: is_peak_surge — surge > 1.5
    # High surge trips may correlate with cancellations and churn
    when(col('surge_multiplier') > 1.5, True).otherwise(False).alias('is_peak_surge'),
)

print(f'  Base feature rows  : {df_trip_base.count():,}')
print(f'  Columns so far     : {len(df_trip_base.columns)}')


Building trip base features...
  Base feature rows  : 21,359,745
  Columns so far     : 23


In [0]:
print('Enriching with driver attributes...')

# Select only needed driver columns before broadcast join
# Keeps join payload small — don't carry all 15 driver columns
df_driver_lookup = df_drivers.select(
    col('driver_id'),
    col('rating').alias('driver_rating'),
    col('status').alias('driver_status'),
    col('vehicle_type').alias('driver_vehicle_type'),
    col('is_high_value_driver'),
)

df_enriched = df_trip_base.join(
    broadcast(df_driver_lookup),
    on='driver_id',
    how='left'
)

# Check how many trips have no matching driver
unmatched_drivers = df_enriched.filter(col('driver_rating').isNull()).count()
print(f'  Trips with no matching driver : {unmatched_drivers:,}')
print(f'  Columns after driver enrich   : {len(df_enriched.columns)}')


Enriching with driver attributes...
  Trips with no matching driver : 0
  Columns after driver enrich   : 27


In [0]:
print('Enriching with customer attributes...')

# Select only needed customer columns before broadcast join
df_customer_lookup = df_customers.select(
    col('customer_id'),
    col('city').alias('customer_city'),
    col('device_type').alias('customer_device_type'),
    col('is_churned'),
)

df_enriched = df_enriched.join(
    broadcast(df_customer_lookup),
    on='customer_id',
    how='left'
)

# Check how many trips have no matching customer
unmatched_customers = df_enriched.filter(col('customer_city').isNull()).count()
print(f'  Trips with no matching customer : {unmatched_customers:,}')
print(f'  Columns after customer enrich   : {len(df_enriched.columns)}')


Enriching with customer attributes...
  Trips with no matching customer : 0
  Columns after customer enrich   : 30


In [0]:
df_gold = df_enriched \
    .withColumn('_processed_at', current_timestamp()) \
    .withColumn('_source', lit('silver.trips+drivers+customers'))

print(f'Final row count   : {df_gold.count():,}')
print(f'Final column count: {len(df_gold.columns)}')
print()
print('Gold schema:')
df_gold.printSchema()


Final row count   : 21,359,745
Final column count: 32

Gold schema:
root
 |-- customer_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- trip_status: string (nullable = true)
 |-- pickup_time: timestamp (nullable = true)
 |-- drop_time: timestamp (nullable = true)
 |-- trip_date: date (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- surge_multiplier: double (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- weather: string (nullable = true)
 |-- day_type: string (nullable = true)
 |-- is_ghost_trip: boolean (nullable = true)
 |-- is_fare_inflated: boolean (nullable = true)
 |-- is_distance_invalid: boolean (nullable = true)
 |-- is_duration_invalid: boolean (nullable = true)
 |-- is_status_invalid: boolean (nullable = true)
 |-- fare_per_km: double (nullable = 

In [0]:
print({LOAD_TYPE})

{'full'}


In [0]:
import time
print(f'Writing gold.trip_features — mode: {LOAD_TYPE}...')
t0 = time.time()

if LOAD_TYPE == 'full':
    df_gold.write \
        .format('delta') \
        .mode('overwrite') \
        .partitionBy('trip_status') \
        .option('overwriteSchema', 'true') \
        .saveAsTable(f'{GOLD}.trip_features')
    print(f'  Full load written : {df_gold.count():,} rows')
    print(f'  Write time        : {time.time()-t0}s')
    print('Running OPTIMIZE + ZORDER...')
    spark.sql(f'OPTIMIZE {GOLD}.trip_features ZORDER BY (trip_date, city)')
    print('  OPTIMIZE + ZORDER complete')
else:
    new_count = df_gold.count()
    if new_count > 0:
        df_gold.write \
            .format('delta') \
            .mode('append') \
            .saveAsTable(f'{GOLD}.trip_features')
        print(f'  Incremental rows appended : {new_count:,}')
        print(f'  Write time                : {time.time()-t0}s')
        print('Running OPTIMIZE + ZORDER...')
        spark.sql(f'OPTIMIZE {GOLD}.trip_features ZORDER BY (trip_date, city)')
        print('  OPTIMIZE + ZORDER complete')
    else:
        print('  No new rows — skipping write and OPTIMIZE')


Writing gold.trip_features — mode: full...
  Full load written : 21,359,745 rows
  Write time        : 24.808610677719116s
Running OPTIMIZE + ZORDER...
  OPTIMIZE + ZORDER complete


In [0]:
print('=== GOLD TRIP FEATURES VERIFICATION ===')
print()

df_verify = spark.table(f'{GOLD}.trip_features')
total = df_verify.count()

print(f'  Total rows    : {total:,}')
print(f'  Total columns : {len(df_verify.columns)}')
print(f'  Load type     : {LOAD_TYPE}')
print()

# Data quality checks
print('Data quality checks:')
checks = [
    ('Null trip_id',            df_verify.filter(col('trip_id').isNull()).count()),
    ('Null customer_id',        df_verify.filter(col('customer_id').isNull()).count()),
    ('Null driver_id',          df_verify.filter(col('driver_id').isNull()).count()),
    ('Null trip_date',          df_verify.filter(col('trip_date').isNull()).count()),
    ('Null fare_amount',        df_verify.filter(col('fare_amount').isNull()).count()),
    ('Null driver_rating',      df_verify.filter(col('driver_rating').isNull()).count()),
    ('Null customer_city',      df_verify.filter(col('customer_city').isNull()).count()),
    ('Negative fare_amount',    df_verify.filter(col('fare_amount') < 0).count()),
    ('Negative distance_km',    df_verify.filter(col('distance_km') < 0).count()),
]

all_passed = True
for check, result in checks:
    status = 'PASS' if result == 0 else 'WARN'
    if result > 0: all_passed = False
    print(f'  {status}  {check:<30} : {result:,}')

print()
print('Trip status distribution:')
df_verify.groupBy('trip_status').count().orderBy('count', ascending=False).show()

print('City distribution:')
df_verify.groupBy('city').count().orderBy('count', ascending=False).show()

print('Vehicle type distribution:')
df_verify.groupBy('vehicle_type').count().orderBy('count', ascending=False).show()

print('Weather distribution:')
df_verify.groupBy('weather').count().orderBy('count', ascending=False).show()

print('Day type distribution:')
df_verify.groupBy('day_type').count().orderBy('count', ascending=False).show()

print('Derived flag summary:')
from pyspark.sql.functions import sum as spark_sum
df_verify.agg(
    spark_sum(when(col('is_long_trip')    == True, 1).otherwise(0)).alias('long_trips'),
    spark_sum(when(col('is_peak_surge')   == True, 1).otherwise(0)).alias('peak_surge_trips'),
    spark_sum(when(col('is_ghost_trip')   == True, 1).otherwise(0)).alias('ghost_trips'),
    spark_sum(when(col('is_fare_inflated')== True, 1).otherwise(0)).alias('fare_inflated_trips'),
    spark_sum(when(col('is_high_value_driver') == True, 1).otherwise(0)).alias('high_value_driver_trips'),
).show(truncate=False)

print('Key feature stats:')
df_verify.select(
    'distance_km', 'fare_amount', 'surge_multiplier',
    'trip_duration_minutes', 'fare_per_km', 'driver_rating'
).summary('min', 'max', 'mean', '50%').show(truncate=False)

print('Sample rows:')
df_verify.select(
    'trip_id', 'city', 'trip_date', 'trip_status',
    'distance_km', 'fare_amount', 'fare_per_km',
    'driver_rating', 'is_high_value_driver',
    'is_long_trip', 'is_peak_surge', 'is_churned'
).limit(5).show(truncate=False)

print()
detail = spark.sql(f'DESCRIBE DETAIL {GOLD}.trip_features').collect()[0]
print(f'  numFiles    : {detail["numFiles"]}')
print(f'  sizeInBytes : {detail["sizeInBytes"]:,}')

print()
if all_passed:
    print('All quality checks passed. Gold trip features ready.')
else:
    print('Some checks flagged — review WARN items above.')
print('Next: UrbanRide_09 — Gold Payment Features')


=== GOLD TRIP FEATURES VERIFICATION ===

  Total rows    : 21,359,745
  Total columns : 32
  Load type     : full

Data quality checks:
  PASS  Null trip_id                   : 0
  PASS  Null customer_id               : 0
  PASS  Null driver_id                 : 0
  PASS  Null trip_date                 : 0
  PASS  Null fare_amount               : 0
  PASS  Null driver_rating             : 0
  PASS  Null customer_city             : 0
  PASS  Negative fare_amount           : 0
  WARN  Negative distance_km           : 64,336

Trip status distribution:
+-----------+--------+
|trip_status|   count|
+-----------+--------+
|  completed|19542216|
|  cancelled| 1817529|
+-----------+--------+

City distribution:
+---------+-------+
|     city|  count|
+---------+-------+
|    Delhi|5794698|
|   Mumbai|5039615|
|Bengaluru|4263697|
|Hyderabad|3524479|
|     Pune|2737256|
+---------+-------+

Vehicle type distribution:
+------------+-------+
|vehicle_type|  count|
+------------+-------+
|       Se

### Derived Flag Summary — Key observations:
| Flag | Count | % of total | Note |
|---|---|---|---|
| long_trips | 81,058 | 0.38% | Fixed from 158 — threshold updated 20km → 15km ✅ |
| peak_surge_trips | 4,210,608 | 19.7% | Nearly 1 in 5 trips hit peak surge — strong fraud/churn signal |
| ghost_trips | 39,067 | 0.18% | Consistent with Silver ✅ |
| fare_inflated | 21,201 | 0.10% | Consistent with Silver ✅ |
| high_value_driver_trips | 6,366,339 | 29.8% | Nearly 30% of trips served by high value drivers |

1. **is_long_trip threshold matters** — 20km flagged only 158 trips out of 21M (near zero).
   Dropping to 15km flagged 81,058 — 

2. **Peak surge is more common than expected** — 19.7% of all trips had surge > 1.5.
   This is a strong candidate feature for both churn and fraud models.

3. **Ghost trips (0.18%) and fare inflated (0.10%) are rare but preserved in Gold** —
   flag-not-drop pattern ensures fraud model sees these anomalies.

4. H**igh value drivers handle 29.8% of trips** — disproportionately high given
   they are only 28.3% of the driver base. Suggests high value drivers
   are more active / available than inactive ones.

5. **Negative distance (64,336 rows) carries through Gold as WARN — not dropped.**
   fare_per_km is null-safe for these rows. Downstream aggregations must
   filter is_distance_invalid == False before using distance_km.